# DS behavior video conversion jobs

Discover `Flir*.avi` files inside `*-good` folders and submit one SLURM job per AVI to convert it to MP4 on the cluster.

In [2]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'miscellaneous' else NOTEBOOK_DIR

for path in (REPO_ROOT, REPO_ROOT / 'miscellaneous'):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from ds_behav_cluster_jobs import (
    DEFAULT_CLUSTER_HOST,
    DEFAULT_CONDA_ENV,
    DEFAULT_LOG_DIR,
    DEFAULT_OUTPUT_DIR,
    DEFAULT_PIPELINE_WORKDIR,
    DEFAULT_RUNNER_SCRIPT,
    DEFAULT_USERNAME,
    connect_ssh,
    discover_flir_avi_jobs,
    print_job_summary,
    submit_conversion_jobs,
    wait_for_jobs,
)


In [3]:
CKII_data_folders = [
    '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce21/2025-08-06_pAce21_PR/Awake',
    '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce38/2025-11-26_pAce38/PX/post/Awake',
    '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce45/2026-01-18_pAce45/PX/post/Awake',
    '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce46/2026-02-22_pAce46/PR/post/Awake',
    '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce47/2026-01-28_pAce47/PX/post/Awake',
    '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce50/2026-03-17_pAce50_PRL/Awake',
]

CLUSTER_HOST = DEFAULT_CLUSTER_HOST
USERNAME = DEFAULT_USERNAME
CONDA_ENV = DEFAULT_CONDA_ENV
PIPELINE_WORKDIR = DEFAULT_PIPELINE_WORKDIR
RUNNER_SCRIPT = DEFAULT_RUNNER_SCRIPT
OUTPUT_DIR = DEFAULT_OUTPUT_DIR
LOG_DIR = DEFAULT_LOG_DIR
FFMPEG_BINARY = 'ffmpeg'

jobs = discover_flir_avi_jobs(CKII_data_folders, output_dir=OUTPUT_DIR)
print_job_summary(jobs)
print(f'\nConda env on cluster: {CONDA_ENV}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Log dir: {LOG_DIR}')


Discovered 10 AVI file(s).
01. /Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce21/2025-08-06_pAce21_PR/Awake/4-good/Flir-08062025102344-0000.avi
    -> /Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/All_behavior_videos_mp4/pAce21_PR_2025-08-06_4_good.mp4
02. /Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce38/2025-11-26_pAce38/PX/post/Awake/2-good/Flir-11262025134541-0000.avi
    -> /Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/All_behavior_videos_mp4/pAce38_PX_2025-11-26_2_good.mp4
03. /Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce38/2025-11-26_pAce38/PX/post/Awake/4-good/Flir-11262025140659-0000.avi
    -> /Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/All_behavior_videos_mp4/pAce38_PX_2025-11-26_4_good.mp4
04. /Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce45/2026-01-18_pAce45/PX/post/Awake/2-good/Flir-01182026135502-0000.avi
    -> /Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/All_behavior_videos_mp4/pAce45_PX_2026-01-18_2_goo

In [5]:
ssh = connect_ssh(cluster_host=CLUSTER_HOST, username=USERNAME)
dry_run_results = submit_conversion_jobs(
    ssh,
    jobs,
    pipeline_workdir=PIPELINE_WORKDIR,
    conda_env=CONDA_ENV,
    runner_script=RUNNER_SCRIPT,
    log_dir=LOG_DIR,
    ffmpeg_binary=FFMPEG_BINARY,
    cpus=2,
    mem_gb=32,
    time_limit='08:00:00',
    crf=17,
    preset='slow',
    overwrite=False,
    dry_run=True,
)

for item in dry_run_results:
    print(item['command'])
    print()


sbatch --job-name flir_mp4_pAce21_4_good -c 2 --mem 32G -t 08:00:00 --export ALL,CONDA_ENV_NAME=adamlab_pipeline --chdir /ems/elsc-labs/adam-y/qixin.yang/ClusterCode/miniVI_Renana_Pipeline --output /ems/elsc-labs/adam-y/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miscellaneous/logs_ds_behav/%x_%j.out --error /ems/elsc-labs/adam-y/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miscellaneous/logs_ds_behav/%x_%j.err /ems/elsc-labs/adam-y/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/utils/run_python_job.sh /ems/elsc-labs/adam-y/qixin.yang/ClusterCode/miniVI_Renana_Pipeline miscellaneous/convert_flir_avi_to_mp4.py --input-video /ems/elsc-labs/adam-y/Adam-Lab-Shared/Data/renana_malka/pAce21/2025-08-06_pAce21_PR/Awake/4-good/Flir-08062025102344-0000.avi --output-video /ems/elsc-labs/adam-y/Adam-Lab-Shared/Data/renana_malka/All_behavior_videos_mp4/pAce21_PR_2025-08-06_4_good.mp4 --ffmpeg-binary ffmpeg --crf 17 --preset slow

sbatch --job-name flir_mp4_pAce38_2_good -c 2 --mem 32G -t 08:00:00

In [6]:
# Run after verifying the dry-run commands above.
submit_results = submit_conversion_jobs(
    ssh,
    jobs,
    pipeline_workdir=PIPELINE_WORKDIR,
    conda_env=CONDA_ENV,
    runner_script=RUNNER_SCRIPT,
    log_dir=LOG_DIR,
    ffmpeg_binary=FFMPEG_BINARY,
    cpus=2,
    mem_gb=8,
    time_limit='08:00:00',
    crf=17,
    preset='slow',
    overwrite=False,
    dry_run=False,
)

for item in submit_results:
    print(item['output'] or item['error'])

job_ids = [item['job_id'] for item in submit_results if item['job_id']]
wait_for_jobs(ssh, job_ids, poll_interval=60)


[14:13:11] Jobs status:
  Completed: 10/10
  Pending:   0
  Failed:    [('32162408', 'FAILED'), ('32162409', 'FAILED'), ('32162410', 'FAILED'), ('32162411', 'FAILED'), ('32162412', 'FAILED'), ('32162413', 'FAILED'), ('32162414', 'FAILED'), ('32162415', 'FAILED'), ('32162416', 'FAILED'), ('32162417', 'FAILED')]


False